In [29]:
import os
import openprotein
from openprotein.protein import Protein
from openprotein.chains import DNA
from dotenv import load_dotenv 

# Please make a .env file with your username and password. The format should be something like:
# OPENPROTEIN_USERNAME= your username (mine was my email)
# OPENPROTEIN_PASSWORD= password
# Load the environment variables from the local .env file
load_dotenv() 

user = os.environ.get("OPENPROTEIN_USERNAME")
pw = os.environ.get("OPENPROTEIN_PASSWORD")

session = openprotein.connect(username=user, password=pw)

In [34]:
# Define the proteins
proteins = [
    # Testing with CRE 
    Protein(sequence="MSNLLTVHQNLPALPVDATSDEVRKNLMDMFRDRQAFSEHTWKMLLSVCRSWAAWCKLNNRKWFPAEPEDVRDYLLYLQARGLAVKTIQQHLGQLNMLHRRSGLPRPSDSNAVSLVMRRIRKENVDAGERAKQALAFERTDFDQVRSLMENSDRCQDIRNLAFLGIAYNTLLRIAEIARIRVKDISRTDGGRMLIHIGRTKTLVSTAGVEKALSLGVTKLVERWISVSGVADDPNNYLFCRVRKNGVAAPSATSQLSTRALEGIFEATHRLIYGAKDDSGQRYLAWSGHSARVGAARDMARAGVSIPEIMQAGGWTNVNIVMNYIRNLDSETGAMVRLLEDGD")
]
proteins[0].chain_id = ["A", "B", "C", "D"]

# Define the ligand
dnas = [
    # 2 LoxP chains
    DNA(sequence="ATAACTTCGTATAATGTATCCTCTATACGAACTTAT", chain_id = "E"),
    DNA(sequence="ATAACTTCGTATAATGTATCCTCTATACGAACTTAT", chain_id = "F")
]

In [35]:
msa_query = []
for p in proteins:
    if p.chain_id is not None and isinstance(p.chain_id, list):
        for _ in p.chain_id:
            msa_query.append(p.sequence.decode())
    else:
        msa_query.append(p.sequence.decode())
msa = session.align.create_msa(seed=":".join(msa_query))

for p in proteins:
    p.msa = msa

In [36]:
fold_job = session.fold.boltz2.fold(
    proteins=proteins,
    dnas=dnas,
)
fold_job

FoldJob(num_records=1, job_id='904b7f1a-0fa7-4b07-af3c-2a25a5a93608', job_type=<JobType.embeddings_fold: '/embeddings/fold'>, status=<JobStatus.PENDING: 'PENDING'>, created_date=datetime.datetime(2025, 10, 16, 19, 29, 30, 870052, tzinfo=TzInfo(0)), start_date=None, end_date=None, prerequisite_job_id=None, progress_message=None, progress_counter=0, sequence_length=None)

In [37]:
fold_job.wait_until_done(verbose=True)

Waiting: 100%|██████████| 100/100 [06:14<00:00,  3.74s/it, status=SUCCESS]


True

In [30]:
# Get the result as a PDB bytestring
result = fold_job.get()

print('\n'.join(result.decode().splitlines()[500:510])) # Print a few lines

ATOM 47 C CA . VAL A 1 7 ? -17.08811 -27.50397 23.01897 1 63.581 ? 7 A 1
ATOM 48 C C . VAL A 1 7 ? -15.72016 -28.00331 23.45939 1 63.581 ? 7 A 1
ATOM 49 O O . VAL A 1 7 ? -15.2476 -27.68237 24.55406 1 63.581 ? 7 A 1
ATOM 50 C CB . VAL A 1 7 ? -18.19451 -28.49313 23.43961 1 63.581 ? 7 A 1
ATOM 51 C CG1 . VAL A 1 7 ? -18.00867 -28.92698 24.88565 1 63.581 ? 7 A 1
ATOM 52 C CG2 . VAL A 1 7 ? -19.55847 -27.84167 23.25925 1 63.581 ? 7 A 1
ATOM 53 N N . HIS A 1 8 ? -15.08352 -28.78162 22.6134 1 64.772 ? 8 A 1
ATOM 54 C CA . HIS A 1 8 ? -13.73669 -29.25524 22.90964 1 64.772 ? 8 A 1
ATOM 55 C C . HIS A 1 8 ? -12.78482 -28.08959 23.11863 1 64.772 ? 8 A 1
ATOM 56 O O . HIS A 1 8 ? -11.83611 -28.17824 23.90363 1 64.772 ? 8 A 1


In [38]:
from molviewspec import create_builder
builder = create_builder()
structure = builder.download(url="mystructure.cif")\
    .parse(format="mmcif")\
    .model_structure()\
    .component()\
    .representation()\
    .color(color="blue")
builder.molstar_notebook(data={'mystructure.cif': result}, width=500, height=400)

<IPython.core.display.Javascript object>

In [39]:
# Retrieve the pLDDT scores
plddt_scores = fold_job.plddt
print("pLDDT scores shape:", plddt_scores.shape)
print("First 10 scores:", plddt_scores[0, :10])

# Retrieve the PAE matrix
pae_matrix = fold_job.pae
print("\nPAE matrix shape:", pae_matrix.shape)

# Retrieve the PDE matrix
pde_matrix = fold_job.pde
print("\nPDE matrix shape:", pde_matrix.shape)

# Retrieve the confidence scores
import json
confidence_scores = fold_job.confidence
print("\nConfidence scores:", json.dumps(confidence_scores[0].model_dump(), indent=2))

pLDDT scores shape: (1, 1444)
First 10 scores: [0.4815873  0.5213046  0.5648581  0.63016856 0.5945839  0.5805369
 0.5855394  0.60746014 0.5869623  0.6195915 ]

PAE matrix shape: (1, 1444, 1444)

PDE matrix shape: (1, 1444, 1444)

Confidence scores: {
  "confidence_score": 0.8606483340263367,
  "ptm": 0.9007336497306824,
  "iptm": 0.8815740346908569,
  "ligand_iptm": 0.0,
  "protein_iptm": 0.88047856092453,
  "complex_plddt": 0.8554169535636902,
  "complex_iplddt": 0.8213931918144226,
  "complex_pde": 0.4612204134464264,
  "complex_ipde": 1.2939977645874023,
  "chains_ptm": {
    "0": 0.9513399600982666,
    "1": 0.9689802527427673,
    "2": 0.9543544054031372,
    "3": 0.9685941934585571,
    "4": 0.8878928422927856,
    "5": 0.88595050573349
  },
  "pair_chains_iptm": {
    "0": {
      "0": 0.9513399600982666,
      "1": 0.9281518459320068,
      "2": 0.8444004058837891,
      "3": 0.8794272541999817,
      "4": 0.8580747842788696,
      "5": 0.863185703754425
    },
    "1": {
     

In [40]:
with open("Cre_2loxP.pdb", "wb") as f:
    f.write(result)